# Compare TRF predictors for subject 01

This notebook runs the same subject through several registered TRF models and stores a small comparison table. The large TRF objects stay in the Eelbrain cache; the CSV here is the human-readable experiment log.

In [1]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import sys
import traceback

import numpy as np
import pandas as pd


def find_pipeline_dir(start=Path.cwd()):
    start = Path(start).resolve()
    candidates = [start, *start.parents, start / 'analysis' / 'trf_pipeline']
    for path in candidates:
        if (path / 'alice_eelbrain_main_experiment.py').exists():
            return path
    raise FileNotFoundError(f'Could not find alice_eelbrain_main_experiment.py from {start}')


PIPELINE_DIR = find_pipeline_dir()
PROJECT_ROOT = PIPELINE_DIR.parents[1]
RESULTS_DIR = PIPELINE_DIR / 'results'

if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from alice_eelbrain_main_experiment import TRF_OPTIONS, alice

print(f'Pipeline dir: {PIPELINE_DIR}')
print(f'Results dir: {RESULTS_DIR}')
print(f'TRF options: {TRF_OPTIONS}')

INFO    :  *** AliceComprehensionEelbrainMain initialized with root /Users/yanyuwoo/Data/bids on 2026-07-20 21:48:57 ***
INFO    :  Using eelbrain 0.43.0a1, mne 1.11.0.
Pipeline dir: /Users/yanyuwoo/Desktop/mcmaster-project/alice-comprehension-neural-prediction/analysis/trf_pipeline
Results dir: /Users/yanyuwoo/Desktop/mcmaster-project/alice-comprehension-neural-prediction/analysis/trf_pipeline/results
TRF options: {'samplingrate': 50, 'data': 'eeg', 'tstart': -0.1, 'tstop': 1.0, 'filter_x': 'continuous'}


## Experiment setup

`word-onset` uses the `time` column in `*~word.pickle` and places a unit impulse at each word onset. The lexical predictors use boolean columns from the same word-level table.

In [2]:
SUBJECT = '01'
RAW = '0.5-20'
EPOCH = 'chapter-1'
INV = ''
ESTIMATOR = 'boosting'

MODEL_NAMES = [
    'gammatone-8',
    'acoustic-onset-8',
    'auditory-gammatone',
    'word-onset',
    'lexical-onset',
    'nonlexical-onset',
    'word-logfreq',
    'word-ngram',
    'word-rnn',
    'word-cfg',
    'gammatone-plus-word',
]

model_table = pd.DataFrame(
    [{'model': name, 'term': alice.models[name]} for name in MODEL_NAMES]
)
model_table

,model,term
0,gammatone-8,gammatone-8
1,acoustic-onset-8,gammatone-on-8
2,auditory-gammatone,gammatone-8 + gammatone-on-8
3,word-onset,word
4,lexical-onset,word-lexical
5,nonlexical-onset,word-nlexical
6,word-logfreq,word-LogFreq
7,word-ngram,word-NGRAM
8,word-rnn,word-RNN
9,word-cfg,word-CFG


## Cache paths

Different model definitions should produce different cache files. If you rerun the exact same model and parameters, Eelbrain may reuse the same cached artifact.

In [3]:
cache_rows = []
for model in MODEL_NAMES:
    try:
        cache_path = alice.load_trf(
            model,
            subject=SUBJECT,
            raw=RAW,
            epoch=EPOCH,
            inv=INV,
            estimator=ESTIMATOR,
            path_only=True,
            **TRF_OPTIONS,
        )
        cache_rows.append({
            'subject': SUBJECT,
            'model': model,
            'term': alice.models[model],
            'cache_path': str(cache_path),
            'cache_exists': Path(cache_path).exists(),
        })
    except Exception as error:
        cache_rows.append({
            'subject': SUBJECT,
            'model': model,
            'term': alice.models.get(model, ''),
            'cache_path': '',
            'cache_exists': False,
            'error': repr(error),
        })

cache_table = pd.DataFrame(cache_rows)
cache_table

,subject,model,term,cache_path,cache_exists
0,01,gammatone-8,gammatone-8,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...,True
1,01,acoustic-onset-8,gammatone-on-8,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...,False
2,01,auditory-gammatone,gammatone-8 + gammatone-on-8,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...,False
3,01,word-onset,word,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...,False
4,01,lexical-onset,word-lexical,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...,False
5,01,nonlexical-onset,word-nlexical,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...,False
6,01,word-logfreq,word-LogFreq,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...,False
7,01,word-ngram,word-NGRAM,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...,False
8,01,word-rnn,word-RNN,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...,False
9,01,word-cfg,word-CFG,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...,False


## Run or load TRFs

This cell can take a while. Each row is fit or loaded independently so one failed model does not stop the whole comparison.

In [4]:
def numeric_array(value):
    if hasattr(value, 'x'):
        value = value.x
    return np.asarray(value, dtype=float)


def summarize_result(result):
    r = numeric_array(result.r)
    h = numeric_array(result.h)
    return {
        'r_min': float(np.nanmin(r)),
        'r_max': float(np.nanmax(r)),
        'r_mean': float(np.nanmean(r)),
        'r_abs_mean': float(np.nanmean(np.abs(r))),
        'n_sensors': int(r.size),
        'h_shape': str(h.shape),
        'finite_r': bool(np.isfinite(r).all()),
        'finite_h': bool(np.isfinite(h).all()),
    }


RUN_MODELS = True

rows = []
for model in MODEL_NAMES:
    cache_path = alice.load_trf(
        model,
        subject=SUBJECT,
        raw=RAW,
        epoch=EPOCH,
        inv=INV,
        estimator=ESTIMATOR,
        path_only=True,
        **TRF_OPTIONS,
    )
    row = {
        'run_time': datetime.now().isoformat(timespec='seconds'),
        'subject': SUBJECT,
        'model': model,
        'term': alice.models[model],
        'estimator': ESTIMATOR,
        'raw': RAW,
        'epoch': EPOCH,
        'tstart': TRF_OPTIONS['tstart'],
        'tstop': TRF_OPTIONS['tstop'],
        'samplingrate': TRF_OPTIONS['samplingrate'],
        'filter_x': TRF_OPTIONS['filter_x'],
        'cache_path': str(cache_path),
        'cache_exists_before': Path(cache_path).exists(),
    }
    if RUN_MODELS:
        try:
            result = alice.load_trf(
                model,
                subject=SUBJECT,
                raw=RAW,
                epoch=EPOCH,
                inv=INV,
                estimator=ESTIMATOR,
                **TRF_OPTIONS,
            )
            row.update(summarize_result(result))
            row['status'] = 'ok'
        except Exception as error:
            row['status'] = 'error'
            row['error'] = repr(error)
            row['traceback'] = traceback.format_exc(limit=3)
    else:
        row['status'] = 'not_run'
    rows.append(row)
    print(f"{model}: {row['status']}")

comparison = pd.DataFrame(rows)
comparison

gammatone-8: ok
Reading 0 ... 366524  =      0.000 ...   733.048 secs...
INFO    :  Raw 0.5-20: filtering for /Users/yanyuwoo/Data/bids/sub-01/eeg/sub-01_task-alice_eeg.vhdr...
Not setting metadata
1 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 1 events and 29271 original time points (prior to decimation) ...
0 bad epochs dropped
Not setting metadata
1 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 1 events and 30924 original time points (prior to decimation) ...
0 bad epochs dropped
Not setting metadata
1 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 1 events and 32131 original time points (prior to decimation) ...
0 bad epochs dropped
Not setting metadata
1 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 1 even

,run_time,subject,model,term,estimator,raw,epoch,tstart,tstop,samplingrate,...,r_max,r_mean,r_abs_mean,n_sensors,h_shape,finite_r,finite_h,status,error,traceback
0,2026-07-20T21:50:17,01,gammatone-8,gammatone-8,boosting,0.5-20,chapter-1,-0.1,1.0,50,...,0.092227,0.032680,0.037916,60.0,"(60, 8, 56)",True,True,ok,NaN,NaN
1,2026-07-20T21:50:17,01,acoustic-onset-8,gammatone-on-8,boosting,0.5-20,chapter-1,-0.1,1.0,50,...,0.087200,0.040322,0.041319,60.0,"(60, 8, 56)",True,True,ok,NaN,NaN
2,2026-07-20T21:50:46,01,auditory-gammatone,gammatone-8 + gammatone-on-8,boosting,0.5-20,chapter-1,-0.1,1.0,50,...,0.094809,0.037551,0.042286,60.0,"(2, 60, 8, 56)",True,True,ok,NaN,NaN
3,2026-07-20T21:52:19,01,word-onset,word,boosting,0.5-20,chapter-1,-0.1,1.0,50,...,0.057876,0.018110,0.020662,60.0,"(60, 56)",True,True,ok,NaN,NaN
4,2026-07-20T21:52:21,01,lexical-onset,word-lexical,boosting,0.5-20,chapter-1,-0.1,1.0,50,...,0.040021,0.017901,0.021474,60.0,"(60, 56)",True,True,ok,NaN,NaN
5,2026-07-20T21:52:23,01,nonlexical-onset,word-nlexical,boosting,0.5-20,chapter-1,-0.1,1.0,50,...,0.045840,0.015269,0.016819,60.0,"(60, 56)",True,True,ok,NaN,NaN
6,2026-07-20T21:52:25,01,word-logfreq,word-LogFreq,boosting,0.5-20,chapter-1,-0.1,1.0,50,...,0.041834,0.020319,0.021777,60.0,"(60, 56)",True,True,ok,NaN,NaN
7,2026-07-20T21:52:28,01,word-ngram,word-NGRAM,boosting,0.5-20,chapter-1,-0.1,1.0,50,...,0.054741,0.021939,0.021992,60.0,"(60, 56)",True,True,ok,NaN,NaN
8,2026-07-20T21:52:30,01,word-rnn,word-RNN,boosting,0.5-20,chapter-1,-0.1,1.0,50,...,0.060999,0.020195,0.021114,60.0,"(60, 56)",True,True,ok,NaN,NaN
9,2026-07-20T21:52:32,01,word-cfg,word-CFG,boosting,0.5-20,chapter-1,-0.1,1.0,50,...,0.040735,0.016578,0.017794,60.0,"(60, 56)",True,True,ok,NaN,NaN


## Save comparison table

The CSV is intentionally small and reviewable. It is the table to use for comparing predictors; the cache path points back to the full TRF object.

In [5]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
output_path = RESULTS_DIR / f'trf_predictor_comparison_sub-{SUBJECT}.csv'

if output_path.exists():
    previous = pd.read_csv(output_path)
    combined = pd.concat([previous, comparison], ignore_index=True, sort=False)
    dedupe_columns = ['subject', 'model', 'estimator', 'raw', 'epoch', 'tstart', 'tstop', 'samplingrate', 'filter_x']
    combined = combined.drop_duplicates(subset=dedupe_columns, keep='last')
else:
    combined = comparison

combined.to_csv(output_path, index=False)
print(f'Wrote {output_path}')
combined.sort_values('r_mean', ascending=False, na_position='last')

Wrote /Users/yanyuwoo/Desktop/mcmaster-project/alice-comprehension-neural-prediction/analysis/trf_pipeline/results/trf_predictor_comparison_sub-01.csv


,run_time,subject,model,term,estimator,raw,epoch,tstart,tstop,samplingrate,...,r_max,r_mean,r_abs_mean,n_sensors,h_shape,finite_r,finite_h,status,error,traceback
1,2026-07-20T21:50:17,01,acoustic-onset-8,gammatone-on-8,boosting,0.5-20,chapter-1,-0.1,1.0,50,...,0.087200,0.040322,0.041319,60.0,"(60, 8, 56)",True,True,ok,NaN,NaN
2,2026-07-20T21:50:46,01,auditory-gammatone,gammatone-8 + gammatone-on-8,boosting,0.5-20,chapter-1,-0.1,1.0,50,...,0.094809,0.037551,0.042286,60.0,"(2, 60, 8, 56)",True,True,ok,NaN,NaN
0,2026-07-20T21:50:17,01,gammatone-8,gammatone-8,boosting,0.5-20,chapter-1,-0.1,1.0,50,...,0.092227,0.032680,0.037916,60.0,"(60, 8, 56)",True,True,ok,NaN,NaN
7,2026-07-20T21:52:28,01,word-ngram,word-NGRAM,boosting,0.5-20,chapter-1,-0.1,1.0,50,...,0.054741,0.021939,0.021992,60.0,"(60, 56)",True,True,ok,NaN,NaN
6,2026-07-20T21:52:25,01,word-logfreq,word-LogFreq,boosting,0.5-20,chapter-1,-0.1,1.0,50,...,0.041834,0.020319,0.021777,60.0,"(60, 56)",True,True,ok,NaN,NaN
8,2026-07-20T21:52:30,01,word-rnn,word-RNN,boosting,0.5-20,chapter-1,-0.1,1.0,50,...,0.060999,0.020195,0.021114,60.0,"(60, 56)",True,True,ok,NaN,NaN
3,2026-07-20T21:52:19,01,word-onset,word,boosting,0.5-20,chapter-1,-0.1,1.0,50,...,0.057876,0.018110,0.020662,60.0,"(60, 56)",True,True,ok,NaN,NaN
4,2026-07-20T21:52:21,01,lexical-onset,word-lexical,boosting,0.5-20,chapter-1,-0.1,1.0,50,...,0.040021,0.017901,0.021474,60.0,"(60, 56)",True,True,ok,NaN,NaN
9,2026-07-20T21:52:32,01,word-cfg,word-CFG,boosting,0.5-20,chapter-1,-0.1,1.0,50,...,0.040735,0.016578,0.017794,60.0,"(60, 56)",True,True,ok,NaN,NaN
5,2026-07-20T21:52:23,01,nonlexical-onset,word-nlexical,boosting,0.5-20,chapter-1,-0.1,1.0,50,...,0.045840,0.015269,0.016819,60.0,"(60, 56)",True,True,ok,NaN,NaN
